# Phase 7 — Stage A: dictionary seeding

Use a Colab GPU runtime. This stage trains a small self-supervised patch autoencoder, clusters 100,000 16×16 patch embeddings into 4,096 unit-normalized atoms, and records the initial coherence. It deliberately does not save generated/prediction images. Encoder and K-Means++ checkpoints/status are written to Google Drive so an interrupted run can resume by rerunning the training cell. The dictionary artifact, JSON metrics, and downloaded DIV2K data are also persisted in Drive. The setup cell clones the latest `master` when needed and fast-forward updates an existing clone.

DIV2K is provided for academic research only; image copyrights remain with their original owners. Enable the download cell only if your use complies with those terms. Optional Flickr2K images can be added from a licensed Drive folder.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/SparseDict')
DATA_ROOT = Path('/content/drive/MyDrive/SparseDict/datasets/DIV2K')
RUN_DIR = Path('/content/drive/MyDrive/SparseDict/phase7/stage_a')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('GPU:', subprocess.run(['nvidia-smi'], check=False, capture_output=True, text=True).stdout[:1000])

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'master', 'https://github.com/kalpeshjp89/SparseDict.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'master'], check=True)
sys.path.insert(0, str(REPO))
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before training.')

In [ ]:
# Set True to download/extract all 800 DIV2K training-HR images (~3.28 GiB archive).
DOWNLOAD_DIV2K = False
if DOWNLOAD_DIV2K:
    subprocess.run([
        sys.executable, str(REPO / 'scripts' / 'download_div2k.py'),
        '--output-dir', str(DATA_ROOT), '--image-count', '800', '--accept-license'
    ], check=True)

In [ ]:
IMAGE_DIRS = [DATA_ROOT / 'DIV2K_train_HR']
# Add licensed, mounted directories when available, for example:
# IMAGE_DIRS.append(Path('/content/drive/MyDrive/datasets/Flickr2K/HR'))
if not all(path.is_dir() for path in IMAGE_DIRS):
    raise FileNotFoundError('Download DIV2K or configure IMAGE_DIRS first.')

command = [
    sys.executable, str(REPO / 'scripts' / 'init_dictionaries.py'),
    '--output-dir', str(RUN_DIR), '--patch-count', '100000',
    '--cluster-count', '4096', '--embedding-dim', '64',
    '--encoder-epochs', '3', '--device', 'cuda'
]
for directory in IMAGE_DIRS:
    command.extend(['--image-dir', str(directory)])
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('Persistent Stage A artifacts:', sorted(path.name for path in RUN_DIR.iterdir()))